# DEAI-opdracht – Lineaire Regressie met Ames Housing

In deze notebook voorspel ik **SalePrice** met een **lineair regressiemodel** op basis van de dataset uit **AmesHousing.xlsx**.

## Mijn gekozen target en top 3 features
- **Target:** `SalePrice`
- **Top 3 verwachte voorspellers:**
  1. `Overall Qual`
  2. `Gr Liv Area`
  3. `Neighborhood` *(categorisch)*

Waarom deze 3?
- `Overall Qual` zegt iets over de algemene kwaliteit van het huis.
- `Gr Liv Area` zegt iets over de woonoppervlakte.
- `Neighborhood` zegt iets over de ligging/wijk, en dat heeft vaak veel invloed op de prijs.


In [ ]:
# Dit blok laadt de libraries die ik nodig heb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)


In [ ]:
# Dit blok leest beide tabjes uit het Excel-bestand in als DataFrames
bestand = "AmesHousing.xlsx"

df = pd.read_excel(bestand, sheet_name="AmesHousing")
data_dictionary = pd.read_excel(bestand, sheet_name="Data Dictionary")

print("Vorm van de dataset:", df.shape)
display(df.head())
display(data_dictionary)


In [ ]:
# Dit blok controleert of elke Excel-kolom goed is ingelezen
print("Kolommen in de DataFrame:")
print(df.columns.tolist())


## Stap 3 – Target en eerste featurekeuze

Uit de Data Dictionary blijkt dat:
- **`SalePrice`** de verkoopprijs is, dus dat is de **targetvariabele**.
- Ik start met deze 3 features:
  - `Overall Qual`
  - `Gr Liv Area`
  - `Neighborhood` *(categorisch, dus deze ga ik one-hot encoden)*

> Extra opmerking: `SGDRegressor` is een lineair regressiemodel dat leert in kleine stapjes. Daardoor kan ik ook experimenteren met dingen zoals **epochs** (`max_iter`) en **learning rate** (`eta0` / `learning_rate`).


In [ ]:
# Dit blok laat zien welke instellingen (hyperparameters) het model heeft
# In Jupyter kun je hiermee de help-functie van Python gebruiken
help(SGDRegressor)


In [ ]:
# Dit blok kiest de eerste features en splitst de data verticaal in X en y
gekozen_features = ["Overall Qual", "Gr Liv Area", "Neighborhood"]
target = "SalePrice"

X = df[gekozen_features]   # Dit zijn de invoerfeatures
y = df[target]             # Dit is de target die ik wil voorspellen

print("Vorm van X:", X.shape)
print("Vorm van y:", y.shape)


In [ ]:
# Dit blok splitst de data horizontaal in train en test
# Zo krijg ik 4 stukken: X_train, X_test, y_train en y_test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


In [ ]:
# Dit blok bepaalt welke kolommen numeriek en categorisch zijn
numerieke_kolommen = X_train.select_dtypes(include=["number"]).columns.tolist()
categorische_kolommen = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerieke kolommen:", numerieke_kolommen)
print("Categorische kolommen:", categorische_kolommen)


In [ ]:
# Dit blok maakt de voorbereiding van de data klaar
# - numerieke kolommen: missende waarden opvullen + schalen
# - categorische kolommen: missende waarden opvullen + one-hot encoden
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerieke_kolommen
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorische_kolommen
        )
    ]
)


In [ ]:
# Dit blok maakt het eerste lineaire regressiemodel aan
# Gekozen hyperparameters:
# - max_iter = aantal leer-rondes / epochs
# - eta0 = begin learning rate
# - learning_rate = manier waarop de learning rate wordt aangepast
eerste_model = SGDRegressor(
    max_iter=1000,
    eta0=0.01,
    learning_rate="invscaling",
    alpha=0.0001,
    penalty="l2",
    random_state=42
)

# Dit blok zet preprocessing en model in één pipeline
eerste_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", eerste_model)
])


In [ ]:
# Dit blok traint het model op de trainingsdata
eerste_pipeline.fit(X_train, y_train)


In [ ]:
# Dit blok maakt voorspellingen op de testdata
eerste_voorspellingen = eerste_pipeline.predict(X_test)

# Dit blok rekent de evaluatiemetrieken uit
eerste_mae = mean_absolute_error(y_test, eerste_voorspellingen)
eerste_mse = mean_squared_error(y_test, eerste_voorspellingen)
eerste_rmse = np.sqrt(eerste_mse)
eerste_r2 = r2_score(y_test, eerste_voorspellingen)

eerste_resultaten = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Waarde": [eerste_mae, eerste_mse, eerste_rmse, eerste_r2]
})

display(eerste_resultaten)


## Stap 7 – Experimenteren

Hieronder maak ik meerdere experimenten.  
Ik verander:
- de **features**
- het aantal **epochs** (`max_iter`)
- de **learning rate** (`eta0` en `learning_rate`)

Ik laat alle resultaten staan, zodat je goed kunt uitleggen hoe je tot je beste model bent gekomen.


In [ ]:
# Dit blok maakt een handige functie om snel nieuwe experimenten te draaien
def run_experiment(naam, features, max_iter, eta0, learning_rate, alpha=0.0001, penalty="l2"):
    # Dit blok kiest X en y
    X = df[features]
    y = df["SalePrice"]

    # Dit blok splitst de data in train en test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Dit blok zoekt numerieke en categorische kolommen
    numerieke_kolommen = X_train.select_dtypes(include=["number"]).columns.tolist()
    categorische_kolommen = X_train.select_dtypes(exclude=["number"]).columns.tolist()

    # Dit blok bereidt de data voor
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numerieke_kolommen
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                categorische_kolommen
            )
        ]
    )

    # Dit blok maakt het lineaire regressiemodel
    model = SGDRegressor(
        max_iter=max_iter,
        eta0=eta0,
        learning_rate=learning_rate,
        alpha=alpha,
        penalty=penalty,
        random_state=42
    )

    # Dit blok bouwt de pipeline
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Dit blok traint het model
    pipeline.fit(X_train, y_train)

    # Dit blok maakt voorspellingen
    voorspellingen = pipeline.predict(X_test)

    # Dit blok rekent de evaluatiemetrieken uit
    mae = mean_absolute_error(y_test, voorspellingen)
    mse = mean_squared_error(y_test, voorspellingen)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, voorspellingen)

    # Dit blok geeft alle belangrijke resultaten terug
    return {
        "Experiment": naam,
        "Features": ", ".join(features),
        "Aantal features": len(features),
        "max_iter": max_iter,
        "eta0": eta0,
        "learning_rate": learning_rate,
        "alpha": alpha,
        "penalty": penalty,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }


In [ ]:
# Dit blok draait meerdere experimenten achter elkaar
experimenten = []

# Initiële run
experimenten.append(
    run_experiment(
        naam="Initiële run",
        features=["Overall Qual", "Gr Liv Area", "Neighborhood"],
        max_iter=1000,
        eta0=0.01,
        learning_rate="invscaling"
    )
)

# Experiment 1: zelfde features, maar andere epochs en learning rate
experimenten.append(
    run_experiment(
        naam="Exp 1: zelfde features, andere hyperparameters",
        features=["Overall Qual", "Gr Liv Area", "Neighborhood"],
        max_iter=2000,
        eta0=0.005,
        learning_rate="adaptive"
    )
)

# Experiment 2: extra numerieke features erbij
experimenten.append(
    run_experiment(
        naam="Exp 2: meer features",
        features=["Overall Qual", "Gr Liv Area", "Neighborhood", "Total Bsmt SF", "Year Built"],
        max_iter=2000,
        eta0=0.005,
        learning_rate="adaptive"
    )
)

# Experiment 3: nog meer features, inclusief extra categorische info
experimenten.append(
    run_experiment(
        naam="Exp 3: nog meer features",
        features=["Overall Qual", "Gr Liv Area", "Neighborhood", "Total Bsmt SF", "Year Built", "Garage", "House Style"],
        max_iter=2000,
        eta0=0.005,
        learning_rate="adaptive"
    )
)

# Experiment 4: bijna alle bruikbare features
experimenten.append(
    run_experiment(
        naam="Exp 4: bijna alle features",
        features=["Garage", "Overall Qual", "Gr Liv Area", "Total Bsmt SF", "Lot Area", "Year Built", "Full Bath", "Bedroom AbvGr", "Neighborhood", "House Style"],
        max_iter=2000,
        eta0=0.005,
        learning_rate="adaptive"
    )
)

# Experiment 5: zelfde features als exp 4, maar andere epochs en learning rate
experimenten.append(
    run_experiment(
        naam="Exp 5: bijna alle features + andere epochs/lr",
        features=["Garage", "Overall Qual", "Gr Liv Area", "Total Bsmt SF", "Lot Area", "Year Built", "Full Bath", "Bedroom AbvGr", "Neighborhood", "House Style"],
        max_iter=3000,
        eta0=0.003,
        learning_rate="adaptive"
    )
)

resultaten_df = pd.DataFrame(experimenten).sort_values("RMSE").reset_index(drop=True)
display(resultaten_df)


In [ ]:
# Dit blok laat alleen de belangrijkste scores netjes zien
samenvatting = resultaten_df[["Experiment", "Aantal features", "max_iter", "eta0", "learning_rate", "MAE", "RMSE", "R2"]].copy()
display(samenvatting.round(3))


In [ ]:
# Dit blok pakt het beste experiment op basis van de laagste RMSE
beste_model = resultaten_df.iloc[0]
display(beste_model)


## Korte conclusie

- De **initiële run** werkte al redelijk goed.
- Alleen de **hyperparameters aanpassen** gaf al een kleine verbetering.
- **Meer relevante features toevoegen** gaf de grootste winst.
- Het **beste model** in deze notebook is:
  - **Exp 4: bijna alle features**
  - met `max_iter = 2000`
  - `eta0 = 0.005`
  - `learning_rate = "adaptive"`

### Waarom inspireerde elk experiment het volgende?
- Omdat Exp 1 iets beter was dan de initiële run, wist ik dat **epochs/learning rate** invloed hadden.
- Daarna heb ik **meer features toegevoegd**, omdat er waarschijnlijk nog bruikbare informatie ontbrak.
- Toen dat weer beter werkte, heb ik bijna alle bruikbare features geprobeerd.
- Daarna testte ik nóg een andere combinatie van epochs/lr (Exp 5), maar die was net iets minder goed dan Exp 4.

### Eindconclusie
Het model werd duidelijk beter:
- **Lagere fout** (MAE en RMSE daalden)
- **Hogere verklaarde variantie** (`R2` steeg)

Dus: **meer goede features + nette hyperparameters = beter lineair regressiemodel**.
